In [0]:
import argparse
import pprint
import os
import datetime as dt

import sys
sys.path.append('..')
sys.path.append('../..')

import yaml
from lib_dna_category.build import Build
from databricks.feature_engineering import FeatureEngineeringClient

In [0]:
%run ../../config/utils

In [0]:
config_path = dbutils.widgets.get("config_path")
reqs_path   = dbutils.widgets.get("reqs_path")

In [0]:
def open_configs(config_path, reqs_path):
    """ """
    with open(config_path) as config_file:
        config = yaml.load(config_file, Loader=yaml.FullLoader)

    with open(reqs_path) as reqs_file:
        reqs = yaml.load(reqs_file, Loader=yaml.FullLoader)

    return (reqs, config)

In [0]:
reqs, config = open_configs(config_path, reqs_path)

category    = config["category"]

table_name  = reqs["out"]["path"] + category.lower() + '_category_dna_full'
# the table names should read something like:  fs_ah4_cd_category_dna_full

df, config = Build.execute(spark, reqs, config, intermediate_all_tables_dict)

In [0]:
fe = FeatureEngineeringClient()
pk_datatype = df.schema[category].dataType
fe.write_table(
    name=globals()[table_name],
    df=df.withColumn(category, f.coalesce(category, f.lit('-1')).cast(pk_datatype)),
    mode="merge"
)

df_archive = spark.table(globals()[table_name]).withColumn('START_DATE', f.lit(config["start"]).cast('date'))
df_archive = df_archive.withColumn('END_DATE', f.lit(config["end"]).cast('date'))
df_archive = df_archive.withColumn('RUN_DATE', f.lit(dt.datetime.now().replace(second=0, microsecond=0)).cast("timestamp"))

df_archive.write.mode("append").saveAsTable(globals()[category.lower() + '_category_dna_archive'])